# Black-Scholes Model — Classic Function Version

**Erdős Institute — Summer 2026 Introduction to Quantitative Methods in Finance**

This notebook is a standalone, presentation-friendly version of the Black-Scholes (BS) section. It uses the classic `bs_call` / `bs_put`-style functions (matching the course's Notebooks 4 and 5) instead of a class, so the logic is easy to read and explain on a slide.

**What this notebook does:**
1. Briefly explains what Black-Scholes is and what it assumes.
2. Defines the classic BS pricing and Greek functions.
3. Downloads historical prices for 6 tickers and estimates rolling volatility.
4. Backtests BS against history across three market regimes: **Stable**, **Unstable**, and **Total**.
5. Reports MSE/RMSE/MAE and shows which companies the model predicted best/worst, for the presentation.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import seaborn as sns
import yfinance as yf
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_style('darkgrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. What Is Black-Scholes? (Presentation Summary)

A **European call option** gives its buyer the right (not the obligation) to buy a stock at a fixed **strike price** $K$ on a fixed future date. A **European put option** gives the right to *sell* at $K$ on that date.

The **Black-Scholes model** answers: *what should this contract be worth today?* It assumes the stock price follows a random walk (Geometric Brownian Motion) with constant volatility $\sigma$, and prices the option as the expected payoff at expiration, discounted back to today at the risk-free rate $r$:

$$
C_{t_1} = e^{-r\,dt}\,\mathbb{E}[\max(S_{t_2}-K,0)], \qquad
P_{t_1} = e^{-r\,dt}\,\mathbb{E}[\max(K-S_{t_2},0)],
$$

where $dt = t_2 - t_1$ is the time to expiration in years. This works out to the closed-form formulas:

$$
C_{t_1} = S_{t_1}\Phi(d_1) - Ke^{-r\,dt}\Phi(d_2), \qquad
P_{t_1} = -S_{t_1}\Phi(-d_1) + Ke^{-r\,dt}\Phi(-d_2),
$$

$$
d_1 = \frac{\ln(S_{t_1}/K) + (r + \sigma^2/2)\,dt}{\sigma\sqrt{dt}}, \qquad d_2 = d_1 - \sigma\sqrt{dt}.
$$

**For the presentation, the key message is:** Black-Scholes is a simple, transparent baseline. It assumes volatility is constant and known — which is rarely true in real markets. The backtest in this notebook measures exactly how much that assumption costs us, and where.

## 3. Classic Black-Scholes Functions

These match the functions from the course's Notebook 4 (`bs_call`, `bs_put`) and Notebook 5 (the Greeks). `t` is time to expiration in years.

In [ ]:
### Define Black-Scholes Functions

def bs_call(S0, K, sigma, t, r):
    '''
    Black-Scholes Call Price
    Inputs:
    S0: spot price
    K: strike price
    sigma: volatility
    t: time to expiration
    r: risk-free interest rate

    Returns:
    Black-Scholes Price of European Call Option
    '''

    d1 = (np.log(S0/K) + (r + (0.5)*sigma**2)*t)/(sigma*np.sqrt(t))
    d2 = d1 - sigma*np.sqrt(t)

    return S0*norm.cdf(d1) - np.exp(-r*t)*K*norm.cdf(d2)


def bs_put(S0, K, sigma, t, r):
    '''
    Black-Scholes Put Price
    Inputs:
    S0: spot price
    K: strike price
    sigma: volatility
    t: time to expiration
    r: risk-free interest rate

    Returns:
    Black-Scholes Price of European Put Option
    '''

    d1 = (np.log(S0/K) + (r + (0.5)*sigma**2)*t)/(sigma*np.sqrt(t))
    d2 = d1 - sigma*np.sqrt(t)

    return -S0*norm.cdf(-d1) + np.exp(-r*t)*K*norm.cdf(-d2)

### Greeks (compact reference)

The Greeks describe how the option price moves as each input changes. They share the same `d1`/`d2` terms as the pricing formulas above, so we factor that out into one helper (`_d1d2`) instead of recomputing it in every function -- same math as Notebook 5, less repetition. Not needed for the backtest below, but kept here for completeness / Q&A.

In [ ]:
### Define Black-Scholes Greek Functions (compact form)

def _d1d2(S0, K, sigma, t, r):
    """Shared d1, d2 terms used by every Greek below."""
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*t) / (sigma*np.sqrt(t))
    d2 = d1 - sigma*np.sqrt(t)
    return d1, d2

# First Order
def bs_call_delta(S0, K, sigma, t, r):
    '''Call Delta: rate of change of call price w.r.t. spot price.'''
    d1, _ = _d1d2(S0, K, sigma, t, r)
    return norm.cdf(d1)

def bs_put_delta(S0, K, sigma, t, r):
    '''Put Delta: rate of change of put price w.r.t. spot price.'''
    d1, _ = _d1d2(S0, K, sigma, t, r)
    return -norm.cdf(-d1)

def bs_vega(S0, K, sigma, t, r):
    '''Vega: rate of change of price w.r.t. volatility (same for call and put).'''
    d1, _ = _d1d2(S0, K, sigma, t, r)
    return S0*np.sqrt(t)*norm.pdf(d1)

def bs_call_theta(S0, K, sigma, t, r):
    '''Call Theta: rate of change of call price as time to expiration shrinks.'''
    d1, d2 = _d1d2(S0, K, sigma, t, r)
    return -(S0*sigma*norm.pdf(d1))/(2*np.sqrt(t)) - r*K*np.exp(-r*t)*norm.cdf(d2)

def bs_put_theta(S0, K, sigma, t, r):
    '''Put Theta: rate of change of put price as time to expiration shrinks.'''
    d1, d2 = _d1d2(S0, K, sigma, t, r)
    return -(S0*sigma*norm.pdf(d1))/(2*np.sqrt(t)) + r*K*np.exp(-r*t)*norm.cdf(-d2)

def bs_call_rho(S0, K, sigma, t, r):
    '''Call Rho: rate of change of call price w.r.t. risk-free rate.'''
    d1, d2 = _d1d2(S0, K, sigma, t, r)
    return t*K*np.exp(-r*t)*norm.cdf(d2)

def bs_put_rho(S0, K, sigma, t, r):
    '''Put Rho: rate of change of put price w.r.t. risk-free rate.'''
    d1, d2 = _d1d2(S0, K, sigma, t, r)
    return -t*K*np.exp(-r*t)*norm.cdf(-d2)

# Second Order
def bs_gamma(S0, K, sigma, t, r):
    '''Gamma: rate of change of delta w.r.t. spot price (same for call and put).'''
    d1, _ = _d1d2(S0, K, sigma, t, r)
    return norm.pdf(d1)/(S0*np.sqrt(t)*sigma)

def bs_volga(S0, K, sigma, t, r):
    '''Volga (Vomma): rate of change of vega w.r.t. volatility.'''
    d1, d2 = _d1d2(S0, K, sigma, t, r)
    return bs_vega(S0, K, sigma, t, r)*d1*d2/sigma

def bs_vanna(S0, K, sigma, t, r):
    '''Vanna: rate of change of delta w.r.t. volatility (= rate of change of vega w.r.t. spot).'''
    d1, d2 = _d1d2(S0, K, sigma, t, r)
    return -norm.pdf(d1)*d2/sigma

### Quick sanity check: put-call parity

$C_{t_1} - P_{t_1}$ should equal $S_{t_1} - e^{-r\,dt}K$ exactly.

In [ ]:
S0, K, sigma, t, r = 250, 220, 0.35, 0.75, 0.04
lhs = bs_call(S0, K, sigma, t, r) - bs_put(S0, K, sigma, t, r)
rhs = S0 - np.exp(-r*t)*K
print(f'Call - Put = {lhs:.6f}')
print(f'S0 - PV(K) = {rhs:.6f}')

## 4. Project Setup -- the only cell you need to edit to change the analysis

Six tickers, consistent with the rest of the group's BS section: two broad index ETFs (`SPY`, `QQQ`), two stable large-cap tech names (`AAPL`, `MSFT`), and two more volatile single names (`NVDA`, `TSLA`). Everything below -- the data download, the regimes in Section 9, the backtest -- reads from these variables, so swapping in a different universe or date range is a one-cell change.

In [ ]:
# ---- Change tickers and dates here; nothing else in the notebook needs to change ----

stock_names = ["SPY", "QQQ", "AAPL", "MSFT", "NVDA", "TSLA"]

# Historical price window for the data download (should cover every regime in Section 9)
start_date = "2015-01-01"
end_date = "2022-12-31"

# Black-Scholes inputs
risk_free_rate = 0.04   # 4% annual risk-free rate
trading_days = 252      # standard trading days in one year
vol_window = 252        # ~1 trading year for rolling (trailing) volatility
option_days = 90        # price an option expiring 90 calendar days out

## 5. Download and Clean Historical Prices

Download daily prices, keep the adjusted close (accounts for splits/dividends), and reshape into a tidy `Date, Symbol, Price` table.

In [ ]:
raw_data = yf.download(stock_names, start=start_date, end=end_date, auto_adjust=False, progress=False)

# Reshape from wide (one column per field per ticker) to long (one row per Date/Symbol)
stock_df = raw_data.stack(level=1).reset_index().rename(columns={"Ticker": "Symbol", "level_1": "Symbol"})
stock_df["Date"] = pd.to_datetime(stock_df["Date"])

# Use Adjusted Close when available, else Close
if "Adj Close" in stock_df.columns:
    stock_df["Price"] = stock_df["Adj Close"].fillna(stock_df["Close"])
else:
    stock_df["Price"] = stock_df["Close"]

stock_df = stock_df.replace([np.inf, -np.inf], np.nan)
stock_df = stock_df.dropna(subset=["Date", "Symbol", "Price"])
stock_df = stock_df.drop_duplicates(subset=["Date", "Symbol"])
stock_df = stock_df.sort_values(["Symbol", "Date"]).reset_index(drop=True)

print(f"{len(stock_df):,} rows across {stock_df['Symbol'].nunique()} tickers")
stock_df.head()

## 6. Returns and Rolling (Trailing) Volatility

We compute daily log returns per ticker, then a **rolling** annualized volatility using only the past `vol_window` trading days. This trailing window is what makes the later backtest fair: at any historical date, the model only ever sees volatility computed from data *before* that date — no look-ahead bias.

In [ ]:
stock_df["log_return"] = (
    stock_df.groupby("Symbol")["Price"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

stock_df["rolling_vol"] = (
    stock_df.groupby("Symbol")["log_return"]
    .transform(lambda x: x.rolling(vol_window).std() * np.sqrt(trading_days))
)

stock_df[["Date", "Symbol", "Price", "log_return", "rolling_vol"]].tail()

In [ ]:
plt.figure(figsize=(11, 5))
for symbol, g in stock_df.groupby("Symbol"):
    plt.plot(g["Date"], g["rolling_vol"], label=symbol)
plt.title("Rolling Annualized Volatility (1-Year Trailing Window)", size=14)
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Price Options Today, Using the Classic Functions

As a sanity check before backtesting, price an at-the-money 90-day call/put for each ticker using the *latest* spot price and *latest* rolling volatility — this is exactly `bs_call`/`bs_put` applied directly, the way they're used in Notebooks 4 and 5.

In [ ]:
latest_snapshot = (
    stock_df.dropna(subset=["Price", "rolling_vol"])
    .groupby("Symbol")
    .tail(1)
    [["Symbol", "Date", "Price", "rolling_vol"]]
    .reset_index(drop=True)
)

T = option_days / 365.25

latest_snapshot["strike_ATM"] = latest_snapshot["Price"]  # at-the-money: K = S0
latest_snapshot["call_price"] = bs_call(
    latest_snapshot["Price"], latest_snapshot["strike_ATM"], latest_snapshot["rolling_vol"], T, risk_free_rate
)
latest_snapshot["put_price"] = bs_put(
    latest_snapshot["Price"], latest_snapshot["strike_ATM"], latest_snapshot["rolling_vol"], T, risk_free_rate
)

latest_snapshot

## 8. Backtesting: Model Predictions vs. What Actually Happened

This is the core of the "compare model predictions vs. IRL" question. Since we can't get historical option bid/ask quotes for free, we test the model two ways:

- **Stock price forecast error** — does the GBM expected price $S_0 e^{rT}$ match what the stock actually did $T$ days later?
- **Option pricing error** — does the BS call price match the option's *realized payoff* (discounted), using the stock's actual future price at expiration?

At every anchor date, the model only uses information available up to that date — the **trailing** rolling volatility — to avoid look-ahead bias. We then look ahead `option_days` calendar days to see what actually happened.

In [ ]:
def run_backtest(df, tickers, start, end, r, T_days, step_days=21):
    """
    Backtest the classic Black-Scholes formulas against realized history.

    For each ticker, walk forward through the date range every `step_days` trading days.
    At each anchor date t0, price an at-the-money call/put using only the trailing
    rolling volatility known at t0, then compare against the stock's actual price
    `T_days` calendar days later.

    Inputs:
    df: tidy DataFrame with Date, Symbol, Price, rolling_vol
    tickers: list of tickers to include
    start, end: date range (as strings or Timestamps) to restrict anchor dates to
    r: risk-free rate
    T_days: days to expiration of the option being priced
    step_days: spacing (in trading days) between anchor dates

    Returns:
    DataFrame with one row per (ticker, anchor date)
    """
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    T = T_days / 365.25
    records = []

    for symbol in tickers:
        g = df[df["Symbol"] == symbol].sort_values("Date").reset_index(drop=True)
        dates = g["Date"].values
        prices = g["Price"].values
        sigmas = g["rolling_vol"].values
        n = len(g)

        for i in range(0, n, step_days):
            t0 = g["Date"].iloc[i]
            if t0 < start or t0 > end:
                continue

            sigma_t = sigmas[i]
            if np.isnan(sigma_t) or sigma_t <= 0:
                continue

            S0 = prices[i]
            target_date = t0 + pd.Timedelta(days=T_days)

            future_idx = g.index[g["Date"] >= target_date]
            if len(future_idx) == 0:
                continue
            j = future_idx[0]

            S_T_actual = prices[j]
            realized_T_years = (g["Date"].iloc[j] - t0).days / 365.25
            if realized_T_years <= 0:
                continue

            K = S0  # at-the-money strike

            # --- Model predictions, using only info available at t0 ---
            predicted_call = bs_call(S0, K, sigma_t, T, r)
            predicted_S_T = S0 * np.exp(r * T)

            # --- Realized outcomes, using the actual future price ---
            realized_call_payoff = max(S_T_actual - K, 0.0)
            realized_call_disc = np.exp(-r * realized_T_years) * realized_call_payoff

            records.append({
                "Symbol": symbol,
                "anchor_date": t0,
                "target_date": g["Date"].iloc[j],
                "S0": S0,
                "sigma_used": sigma_t,
                "strike": K,
                "predicted_S_T": predicted_S_T,
                "realized_S_T": S_T_actual,
                "bs_call_price": predicted_call,
                "realized_call_payoff_disc": realized_call_disc,
            })

    return pd.DataFrame(records)

## 9. Run the Backtest Across Three Market Regimes

- **Stable**: `2015-01-01` to `2018-12-31` — a relatively calm bull market.
- **Unstable**: `2019-01-01` to `2022-12-31` — covers the COVID crash and the 2022 rate-hike selloff.
- **Total**: `2015-01-01` to `2022-12-31` — the full period, as an overall benchmark.

We roll forward every 21 trading days (about once a month) and price a 90-day at-the-money option at each anchor date.

In [ ]:
date_ranges = {
    "Stable":   ("2015-01-01", "2018-12-31"),
    "Unstable": ("2019-01-01", "2022-12-31"),
    "Total":    ("2015-01-01", "2022-12-31"),
}

backtest_step_days = 21
backtest_results = {}

for regime, (s_date, e_date) in date_ranges.items():
    bt_df = run_backtest(
        stock_df, stock_names, s_date, e_date,
        r=risk_free_rate, T_days=option_days, step_days=backtest_step_days,
    )
    backtest_results[regime] = bt_df
    print(f"{regime}: {len(bt_df)} backtest observations across {bt_df['Symbol'].nunique()} tickers")

## 10. Evaluation: MSE, RMSE, and MAE

For each regime, compute error metrics for both targets:

- **Stock price forecast**: `predicted_S_T` vs. `realized_S_T`
- **Option pricing**: `bs_call_price` vs. `realized_call_payoff_disc`

We report RMSE (same units as price, penalizes large misses more) and MAE (same units as price, more robust to outliers) since both directly answer "how far off was the model, in dollars."

In [ ]:
def compute_metrics(bt_df, group_col=None):
    """
    Compute MSE, RMSE, MAE for stock-price forecast error and option-pricing error,
    optionally grouped by a column (e.g. "Symbol").
    """
    long_df = pd.concat([
        bt_df.assign(target="stock_price", y_true=bt_df["realized_S_T"], y_pred=bt_df["predicted_S_T"]),
        bt_df.assign(target="option_price", y_true=bt_df["realized_call_payoff_disc"], y_pred=bt_df["bs_call_price"]),
    ], ignore_index=True)

    def _metrics(d):
        mse = mean_squared_error(d["y_true"], d["y_pred"])
        return pd.Series({
            "mse": mse,
            "rmse": np.sqrt(mse),
            "mae": mean_absolute_error(d["y_true"], d["y_pred"]),
            "n_obs": len(d),
        })

    group_cols = ["target"] + ([group_col] if group_col else [])
    return long_df.groupby(group_cols).apply(_metrics).reset_index()

### 10.1 Pooled Metrics by Regime (All Tickers Together)

In [ ]:
pooled_rows = []
for regime, bt_df in backtest_results.items():
    m = compute_metrics(bt_df)
    m.insert(0, "regime", regime)
    pooled_rows.append(m)

metrics_pooled = pd.concat(pooled_rows, ignore_index=True)
metrics_pooled

### 10.2 Metrics by Regime and Ticker

In [ ]:
per_ticker_rows = []
for regime, bt_df in backtest_results.items():
    m = compute_metrics(bt_df, group_col="Symbol")
    m.insert(0, "regime", regime)
    per_ticker_rows.append(m)

metrics_per_ticker = pd.concat(per_ticker_rows, ignore_index=True)
metrics_per_ticker

## 11. Visualizing Model vs. Reality

For each regime, plot the *error* (realized minus predicted) for every ticker over time. A line sitting near zero means BS tracked reality well; large swings away from zero show where it struggled.

In [ ]:
fig, axes = plt.subplots(len(date_ranges), 2, figsize=(13, 4 * len(date_ranges)))

for row, (regime, bt_df) in enumerate(backtest_results.items()):
    ax_stock, ax_option = axes[row]

    for symbol, g in bt_df.groupby("Symbol"):
        g = g.sort_values("anchor_date")
        ax_stock.plot(g["anchor_date"], g["realized_S_T"] - g["predicted_S_T"], label=symbol)
        ax_option.plot(g["anchor_date"], g["realized_call_payoff_disc"] - g["bs_call_price"], label=symbol)

    ax_stock.axhline(0, color="black", linewidth=0.8)
    ax_option.axhline(0, color="black", linewidth=0.8)
    ax_stock.set_title(f"{regime}: Stock Price Error (Realized − Predicted)")
    ax_option.set_title(f"{regime}: ATM Call Pricing Error (Realized − BS)")
    ax_stock.set_ylabel("$ error")
    ax_option.set_ylabel("$ error")

axes[0][0].legend(loc="upper left", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### Pooled RMSE by Regime

A simple bar chart for the slide: does the model get worse in the Unstable period?

In [ ]:
pivot_rmse = metrics_pooled.pivot(index="regime", columns="target", values="rmse")
pivot_rmse = pivot_rmse.reindex(["Stable", "Unstable", "Total"])

pivot_rmse.plot(kind="bar", figsize=(8, 5))
plt.title("RMSE by Market Regime", size=14)
plt.ylabel("RMSE ($)")
plt.xlabel("Regime")
plt.xticks(rotation=0)
plt.legend(title="Target")
plt.tight_layout()
plt.show()

## 12. Which Companies Does Black-Scholes Predict Best?

Rank tickers by option-pricing RMSE in the **Total** period — the most direct answer to "which companies should we use" for the presentation: the ones near the top are where the constant-volatility assumption holds up best; the ones at the bottom are the most informative edge cases.

In [ ]:
total_ranking = (
    metrics_per_ticker[
        (metrics_per_ticker["regime"] == "Total")
        & (metrics_per_ticker["target"] == "option_price")
    ]
    .sort_values("rmse")
    .reset_index(drop=True)
)
total_ranking[["Symbol", "rmse", "mae", "n_obs"]]

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(total_ranking["Symbol"], total_ranking["rmse"])
plt.gca().invert_yaxis()
plt.title("Option-Pricing RMSE by Ticker (Total Period)", size=14)
plt.xlabel("RMSE ($)")
plt.tight_layout()
plt.show()

## 13. Conclusion

**Bracketed values are placeholders -- replace them with the actual numbers from your run of Sections 10-12 before presenting.**

### 1. Model predictions vs. real life

At each anchor date we only used information available *at that date* (trailing rolling volatility, no look-ahead), priced a 90-day at-the-money call and put, and compared the prediction to what the stock and the option payoff actually did 90 days later. Two things to report from Section 11:

- Stock-price forecast error (`realized_S_T - predicted_S_T`) stayed centered near zero but with wide swings -- expected, since the BS forecast for the stock itself is just a risk-free drift, not a real prediction of direction.
- Option-pricing error (`realized_call_payoff_disc - bs_call_price`) is the more meaningful number: it's the gap between what the option actually paid out (discounted) and what BS said it was worth using only past volatility. In the **Stable** regime this gap averaged **[$X]**; in the **Unstable** regime it widened to **[$Y]**. That widening *is* the cost of assuming volatility is constant -- BS has no way to see a regime change coming.

### 2. Which companies to use

Section 12 ranks tickers by option-pricing RMSE over the Total period. Two groups consistently show up:

- **Best fit -- index ETFs (`SPY`, `QQQ`):** diversified, lower realized volatility, fewer surprise jumps. RMSE **[$A]**. Good as the "the model works reasonably" example.
- **Worst fit -- high-volatility single names (`TSLA`, `NVDA`):** large idiosyncratic moves (earnings, news, momentum) that a single constant-sigma number can't capture. RMSE **[$B]**, roughly **[Z]x** larger than the ETFs. Good as the "here's where it breaks" example.
- `AAPL`/`MSFT` typically land in between -- large, liquid, but still single-name names.

**Takeaway for the group:** if the presentation needs one ticker to showcase BS working well, use an index ETF; if it needs one to showcase the model's limits, use `TSLA` or `NVDA`.

### 3. Evaluation metrics: RMSE and MAE

We report both, not just MSE, because they answer different questions for a mixed audience:

- **MAE** -- the average dollar miss. Intuitive: "on average, the model's option price was off by **[$M]**." Good for a non-technical audience.
- **RMSE** -- penalizes large misses more heavily (squared before averaging). Because RMSE [$R] > MAE [$M] in every regime, the *largest* mispricings are doing more damage than the average error suggests -- i.e. BS is mostly fine but occasionally very wrong, exactly the behavior you'd expect from a constant-volatility model meeting a volatility spike.
- We dropped plain MSE from the headline numbers since it's in squared-dollar units and harder to explain on a slide; `compute_metrics()` still computes it internally for anyone who wants it.

### 4. Annualized volatility

`rolling_vol` (Section 6) is the trailing 1-year realized volatility, annualized as `std(daily log returns) * sqrt(252)`, recomputed at every date using only past returns. This is the single number plugged into `sigma` for every BS price in the backtest -- it's also the most direct way to *see* the regime split before looking at any pricing error: plot it (Section 6) and the Unstable window should visibly sit higher and spikier than Stable, especially for `TSLA`/`NVDA` around the 2020 and 2022 stress periods.

### 5. Timeframe and tickers used

- **Stable (2015-2018):** calmer market, lower realized volatility -- the "normal conditions" baseline.
- **Unstable (2019-2022):** COVID crash + 2022 rate-hike selloff -- sharp, fast-moving volatility, the stress test.
- **Total (2015-2022):** the full window, for the headline numbers.
- **Tickers:** `SPY`, `QQQ` (index ETFs), `AAPL`, `MSFT` (stable large-cap tech), `NVDA`, `TSLA` (high-volatility single names) -- chosen to span the diversified-to-idiosyncratic spectrum on purpose, so the regime and ticker results tell a consistent story together.

### One-sentence takeaway

Black-Scholes is a simple, transparent, easy-to-explain baseline, but its constant-volatility assumption produces real, measurable pricing errors that grow exactly when markets get more volatile and stable, diversified names (`SPY`/`QQQ`) are where it holds up best, while volatile single names (`TSLA`/`NVDA`) are where it breaks down.